# Data coverage and SQLite

In [ ]:
from pathlib import Path
import sys

# Works whether Jupyter starts in the project root or in notebooks/.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "config.json").exists() and (p / "src").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from src.data import read_config, download_data
config = read_config(ROOT)


In [ ]:
# One row = one trading session. Dates and source parameters are fixed in config.json.
data = download_data(ROOT, config)
from src.data import audit_coverage, write_sqlite
quality = audit_coverage(data, config["benchmark"])
quality

In [ ]:
columns = ["open", "high", "low", "close", "adj_close", "volume"]
data["MU"][columns].head().round(4)

In [ ]:
import sqlite3
write_sqlite(data, ROOT / "data/market.sqlite")
query = """
SELECT ticker, MIN(date) AS first_date, MAX(date) AS last_date, COUNT(*) AS rows
FROM daily_bars
GROUP BY ticker
ORDER BY ticker
"""
with sqlite3.connect(ROOT / "data/market.sqlite") as connection:
    coverage_sql = pd.read_sql_query(query, connection)
coverage_sql